In [1]:
pip install opencv-python numpy

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
import cv2
import numpy as np

# Initial HSV range
lower_color = np.array([0, 0, 0])
upper_color = np.array([179, 255, 255])

selected = False

# Mouse Click Function
def pick_color(event, x, y, flags, param):
    global lower_color, upper_color, selected

    if event == cv2.EVENT_LBUTTONDOWN:
        frame = param

        # Convert frame to HSV
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # Get clicked pixel color
        pixel = hsv[y, x]

        # Convert to int (avoid overflow)
        h = int(pixel[0])
        s = int(pixel[1])
        v = int(pixel[2])

        # Create HSV range
        lower_color = np.array([
            max(h - 10, 0),
            max(s - 50, 0),
            max(v - 50, 0)
        ])

        upper_color = np.array([
            min(h + 10, 179),
            min(s + 50, 255),
            min(v + 50, 255)
        ])

        selected = True

        print("Selected HSV:", (h, s, v))
        print("Lower Range:", lower_color)
        print("Upper Range:", upper_color)

# Load Video
cap = cv2.VideoCapture("C:/Users/Abror/Desktop/video.mp4")

if not cap.isOpened():
    print("Error: Cannot open video")
    exit()

# Main Loop
while True:
    ret, frame = cap.read()

    if not ret:
        print("End of video")
        break

    # Resize frame
    frame = cv2.resize(frame, (640, 480))

    # Show original video
    cv2.imshow("Video", frame)

    # Enable mouse click
    cv2.setMouseCallback("Video", pick_color, frame)

    if selected:
        # Convert frame to HSV
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # Create mask
        mask = cv2.inRange(hsv, lower_color, upper_color)

        # Noise Removal
        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.erode(mask, kernel, iterations=1)
        mask = cv2.dilate(mask, kernel, iterations=2)

        # STEP 5: Contour Detection
        contours, _ = cv2.findContours(
            mask,
            cv2.RETR_TREE,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for cnt in contours:
            area = cv2.contourArea(cnt)

            # Ignore very small objects (noise)
            if area > 1000:

                # STEP 6: Bounding Box
                x, y, w, h = cv2.boundingRect(cnt)

                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    (0, 255, 0),
                    2
                )

                # STEP 7: Centroid Tracking
                M = cv2.moments(cnt)

                if M["m00"] != 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])

                    # Draw center point
                    cv2.circle(
                        frame,
                        (cx, cy),
                        5,
                        (0, 0, 255),
                        -1
                    )

                    # Show coordinates
                    cv2.putText(
                        frame,
                        f"({cx}, {cy})",
                        (cx + 10, cy),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (255, 255, 255),
                        2
                    )

        # Show mask
        cv2.imshow("Mask", mask)

    # Show final tracking output
    cv2.imshow("Tracking", frame)

    # Press ESC to exit
    if cv2.waitKey(70) & 0xFF == 27:
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()

Selected HSV: (140, 255, 3)
Lower Range: [130 205   0]
Upper Range: [150 255  53]
Selected HSV: (113, 63, 114)
Lower Range: [103  13  64]
Upper Range: [123 113 164]
Selected HSV: (0, 0, 255)
Lower Range: [  0   0 205]
Upper Range: [ 10  50 255]
Selected HSV: (126, 255, 5)
Lower Range: [116 205   0]
Upper Range: [136 255  55]
Selected HSV: (112, 51, 131)
Lower Range: [102   1  81]
Upper Range: [122 101 181]
Selected HSV: (60, 255, 255)
Lower Range: [ 50 205 205]
Upper Range: [ 70 255 255]
Selected HSV: (87, 94, 30)
Lower Range: [77 44  0]
Upper Range: [ 97 144  80]
Selected HSV: (128, 15, 132)
Lower Range: [118   0  82]
Upper Range: [138  65 182]
Selected HSV: (126, 17, 152)
Lower Range: [116   0 102]
Upper Range: [136  67 202]
Selected HSV: (120, 31, 142)
Lower Range: [110   0  92]
Upper Range: [130  81 192]
Selected HSV: (112, 47, 125)
Lower Range: [102   0  75]
Upper Range: [122  97 175]
Selected HSV: (168, 15, 87)
Lower Range: [158   0  37]
Upper Range: [178  65 137]
Selected HSV: (